# N167 proposal: long BDD forecasting and health-aware rate control

Status: designed, not launched. The user confirmed correlated frames rather than IID images. Use 100,000 distinct temporal frames from BDD videos, including 10,000 offline-development frames and 90,000 online arrivals. This is different from BDD100K's official package of one labeled image per video.


## Questions and rationale

The primary forecasting question is whether current/past information predicts the next unseen image's reconstruction loss and next-arrival representation health better than persistence. The control question is whether that information improves the actual learning trajectory at a fixed rate budget. Linear-probe accuracy is a secondary test of representation usefulness.

N166's four matched comparisons favored frozen preview over random by 0.000465 mean incoming loss, but favored periodic over frozen preview by 0.000442. Adaptive action effects were worse than frozen effects in every comparison. Pooled H16 absolute-loss RMSE was 0.06598 frozen and 0.04798 adaptive, versus respective persistence RMSE 0.07921 and 0.07966. Those forecasts concern a 16-image mean. The separate scalar H1 forecaster improved next-image RMSE only about 0.7% over last-loss persistence.

Consequently, retain periodic as a full baseline; change the main forecast target to genuine H1; keep deployed action-effect coefficients fixed after offline development; and allow factual level calibration online. Do not infer useful action selection from lower absolute forecast RMSE alone. N97 motivates rich, action-conditioned features and a persistent offline prior, but its immediate bank/J target and development-label features do not establish next-image prediction and are not imported.


## Data and repeated runs

- 100,000 unique source frames total, sampled as 250 chronological frames from each of 400 distinct videos. Target approximately 15 frames/second with recorded source timestamps, giving about 16.6 seconds per clip. Different clips are separate drives; do not claim cross-clip continuity.
- 6,000 fit frames from 24 videos; 2,000 calibration frames from eight videos; 2,000 locked forecast-test frames from eight videos; 90,000 online frames from 360 videos. Split by whole video before decoding. Calibration and forecast-test results do not update fitted action-effect heads. Factual level adaptation during each evaluation is permitted only after its own issued forecast matures and is reported alongside the frozen shadow.
- Sample across daytime/night and city street/highway/residential, including available weather diversity. Balance the six lighting-by-scene cells as closely as divisibility permits. This is a designed development distribution, not a representative estimate of the entire dataset.
- Use a fixed whole-clip hash order, preserving within-clip chronology. Do not recreate the old repeated five-day/five-night pattern. Metadata determine allocation only, not online predictor features. Report within-clip and clip-boundary errors separately.
- Exclude warm-checkpoint video IDs, evaluation/probe videos and existing development clips where their provenance is known. Save IDs, original splits, source timestamps, hashes and exclusion evidence. Clip disjointness does not establish independent capture sessions. Do not describe reused or unverifiable data as untouched confirmation.
- Two learner/action random seeds, same frozen online sequence and common initial checkpoint: four policy types times two seeds equals eight 90,000-arrival trajectories, or 720,000 arrival exposures. The 100,000 images are a shared unique input pool, not 100,000 newly sampled images per policy.
- Every deployment learner starts from the same existing warm RGB64 MAE and fresh AdamW. Offline learner branches supply predictor training examples; their updated learner weights do not initialize deployment. The first 128 online arrivals fill the recent window with common quarter-rate updates; primary paired comparisons and health forecasting start after that prefix.
- Hold out an additional fixed, video-disjoint labeled keyframe evaluation set from the official validation split: 600 linear-probe support images and 1,800 query images, balanced across the six cells. These are evaluation-only and additional to the 100,000-frame learning/development pool. Neither support nor query images enter the learner, predictor features, controller or replay window. Check both video IDs and exact pixels against warm/offline/online sources before launch.

This temporal design needs more eligible videos than the recent local cohort. The machine currently has all 70k/10k/20k official keyframe splits, a 144-video qualified cohort and additional cached videos, but it does not already have a verified fresh 400-video allocation. Intake is an explicit launch prerequisite. Build a new manifest rather than repeat the recent 15k streams until the arrival counter reaches 100k.


## Four learner policies

| Policy | Decision rule | Purpose |
|---|---|---|
| Periodic | Repeat the fixed nominal/nominal/quarter/nominal/nominal/nominal/skip/nominal/nominal/quarter/nominal/nominal pattern | Strong simple comparator from N166 |
| Random | Sample proportionally to remaining rate counts in the same 12-opportunity block | Separate prediction-guided timing from rate exposure |
| H1 loss controller | Allocate rates using frozen preview-based predictions of next-image loss relative to quarter | Test action selection for the requested H1 target |
| H1 loss + RankMe controller | Same predictor and allocator, adding a half-weight normalized RankMe benefit | Isolate the effect of a health-aware objective |

At every 16th post-fill arrival consider {skip, quarter, nominal}, with nominal LR 0.0001. Quarter is 0.000025. Train only the current image. All intervening arrivals use quarter. The replay/feature window is at most the last 128 arrivals, and stored past images are observations only: no replay training is introduced.

Each complete block of 12 opportunities assigns exactly one skip, two quarter and nine nominal actions. Incomplete trailing blocks use quarter. On 90,000 arrivals including the 128-image prefix, there are 5,617 opportunities, 468 complete blocks and one trailing opportunity: 468 skips, 4,212 nominal steps and 85,320 quarter steps per policy. Thus all four arms have the same update count and LR exposure. Report their measured compute separately; LR is not a FLOP or energy measure.

Use the N166 remaining-quota allocator, frozen offline action prices and temperature, and a 20% mixture with quota-proportional random sampling for learned controllers. Record issued forecasts, sampling probabilities, remaining counts and fallback reasons before acting. The periodic and random controls do not consult predictions. When a context fails the predeclared training-support check (more than one quarter of standardized feature coordinates outside +/-5), use the earliest still-available action in the fixed periodic template. Do not demand strict predicted improvement in both metrics; that previously caused universal skipping and would conflict with the matched quota.


## Exact H1 targets and timeline

At arrival t, observe x_t under learner theta_t before updating it. Forecast candidate outcomes using only x_t, stored past information and the proposed current action a. Execute at most one actual update to obtain theta_t^a. When x_(t+1) arrives, measure the targets under that learner before training on x_(t+1). Score the previously issued forecast, then update the forecaster. The next image is never an input at forecast time.

Primary loss target:

    L_(t+1)^a = fixed-mask reconstruction loss(theta_t^a, x_(t+1)).

Identity-derived masks and observation preprocessing are common across policies. Plot this actual incoming loss separately from the augmented loss of each image actually trained on. Primary prediction errors are H1 errors, not in-sample training loss, a current-image post-update assay, or an H16 average.

At each decision origin, select 63 unique, temporally spaced, strictly past images from the last 128 arrivals, denoted B_t. Measure current health on B_t plus x_t. The H1 health target, revealed on the following arrival, is:

    R_(t+1|t)^a = RankMe(embeddings(theta_t^a, B_t union {x_(t+1)})).

Re-encode every image in that panel under the same current encoder. Do not combine embeddings saved under different encoder versions. The 63 past IDs stay fixed from forecast issuance through scoring. This is actual RankMe of a specified 64-image matrix, not RankMe of a single image or the hypothetical embedding of an unseen image. Log raw variance on the identical matrix as another health diagnostic.

Also record same-membership RankMe on B_t before and after the action, and distinguish this from replacement of x_t by x_(t+1). Offline candidate forks use the same future image and same B_t for all actions. Online, only the chosen candidate's outcome exists. The different candidates' unchosen health and loss are predictions, never measured labels.

After feedback, discard the origin panel as a target definition. It is not a persistent bank older than 128 arrivals. Health H1 forecasts are evaluated at the 5,617 decision opportunities per trajectory; loss forecasts additionally cover every post-fill arrival with an inexpensive predictor. Give these cohorts separate sample counts and RMSE curves.


## Predictors and learning

The low-cost predictor has 14 label-free inputs: current pre-update loss; recent-16 mean and standard deviation; last loss difference; recent-16 loss slope; current brightness and pixel standard deviation; difference from the preceding image; three RGB channel means; most recently measured RankMe; most recently measured log variance; and age of that health measurement. Use only causal history and training-only standardization.

At decision opportunities, add four current gradient-preview features: augmented training-view loss, gradient norm, gradient/momentum cosine and proposed AdamW displacement. This gives the 18-input controller predictor. Refresh the 64-image health measurement at those opportunities. Accepted actions reuse the preview gradient; skip discards it. There are no candidate learner copies or extra branch optimizer updates online. Baseline arms compute the same preview only as a measurement for paired shadow-forecast comparisons, and its cost is reported separately from control/training compute.

Fit regularized action-conditioned H1 loss and log-RankMe heads from offline paired examples for skip, quarter and nominal. Fit the cheap loss head on all offline factual transitions and the paired cases. Offline-only branches may copy states to obtain fair action contrasts. To increase the paired sample count, evaluate such branches every fourth available offline arrival, while the behavior learner follows the predeclared periodic/random rate mixture at its ordinary 16-arrival opportunities. No additional action is applied to the behavior learner during a counterfactual assay.

Represent a candidate forecast as a shared level plus an action effect relative to quarter. Fit effects, scalers and regularization on the 6k fit frames using whole-video internal folds. Use the 2k calibration frames for shrinkage toward mean effects, actor prices/temperature and error calibration. Open the final 2k forecast test only after those choices are fixed; report it without changing the design. Calibrated intervals are empirical diagnostics, not guaranteed coverage under future drift.

During deployment, freeze action-effect feature coefficients. Update the shared level's feature coefficients with the persistent offline ridge prior and a predeclared 1,024-label forgetting half-life, from the actual chosen outcome minus its fixed predicted effect. That is factual residual learning with a model-based decomposition; it is not an observed quarter counterfactual. All positive actions and skips have valid next-arrival outcomes. The cheap loss level learns after every arrival; rich loss and RankMe levels learn after their decision-point targets mature. Their shared level cancels from the action-effect scores, so level adaptation can improve forecasts without silently changing effect rankings.

Every policy runs the same inexpensive shadow forecast comparisons: last-value persistence, scalar AR(1), offline-frozen contextual forecast and contextual forecast with online level learning. At decision opportunities, compare cheap and preview forecasts on exactly the same chosen actions, states and outcomes. Include a separately trained 8-coordinate embedding-projection shadow as an exploratory N97-inspired diagnostic; it cannot affect actions. The projection and scaler are fit only on offline fit embeddings. No task labels enter these predictors.


## Objectives

Let dL(a) be predicted H1 loss minus the quarter forecast, and dR(a) predicted log-RankMe minus the quarter forecast. Fix positive robust scales sL and sR from offline fit contrasts, with numerical floors chosen before evaluation.

    loss score(a) = dL(a) / sL
    joint score(a) = dL(a) / sL - 0.5 * dR(a) / sR

Smaller scores are preferred. Subtract each action's offline-calibration mean score as its fixed opportunity price before quota allocation, so the controller allocates scarce rates to promising contexts rather than only relearning that a larger LR has a different average effect. This is a one-step direct forecast controller. There is no two-step rollout, no imagined future-image sequence and no promise of monotone realized loss or RankMe on each image.

The half weight is a weight on normalized effects, not half a raw RankMe unit. Both learned arms have the same predictor features, fitting data, quota allocator, temperature recipe and online level adaptation. Their controlled difference is the RankMe term.


## Outcomes, evaluation and decision criteria

Primary forecasting outcomes:

- H1 loss RMSE, MAE and squared-error skill relative to last-image persistence and scalar AR(1), on identical observed targets. Separate full-stream cheap forecasts and decision-point rich forecasts.
- H1 log-RankMe RMSE/MAE, raw RankMe RMSE, variance errors and interval coverage versus the corresponding last-measured-value baselines. Exclude warm-fill targets uniformly. Record every forecast before its target is known.
- Whole-run and last-third summaries, rolling 2k-arrival curves, lighting/scene and clip-interior/boundary breakdowns. The last-third summary is fixed in advance, not chosen after viewing curves. Also log the actual following-16-image mean after each decision as a secondary delayed-effect diagnostic; do not relabel its error as H1 error.

Primary achieved learning/health outcomes:

- Mean incoming pre-update reconstruction loss over post-fill arrivals and separately over the final 30k arrivals; measured local RankMe over matched origin IDs; and broader fixed-audit RankMe.
- Fixed evaluation-only audit of 512 query images at start and every 5,000 arrivals, re-encoded under the current encoder. Report raw RankMe, normalized RankMe, feature variance and audit reconstruction loss. Report local and audit banks separately: a temporally narrow panel cannot establish global representation quality.
- At each common checkpoint, relate local RankMe movement to audit RankMe and probe accuracy; do not assume their association from the RankMe literature transfers to this MAE and stream.

Secondary linear probe: at start and every 10k arrivals through 90k, freeze the encoder and refit the same L2-regularized three-class scene classifier on the 600 support embeddings. Fix regularization and preprocessing before the run; fit preprocessing on support only. Query images never train the probe or select its parameters. Report macro balanced accuracy, macro F1 and day/night-conditioned accuracy on the same 1,800 query images. Probe outcomes never change actions, forecasts, stopping time, checkpoints or model selection.

Useful prediction requires positive held-out/prequential skill, not just visually aligned curves. Useful control requires improvement over the periodic baseline as well as random at the identical rate budget. A health-controller success should improve local and audit RankMe without a loss penalty, with probe performance providing a secondary check; show the loss/health trade-off if it does not dominate. Report seed-specific effects and paired clip/block resampling intervals with their serial-dependence limitations. Two seeds and one clip order are development evidence, not broad generalization.


## Runtime, storage and notebook

Use a streaming RGB64 uint8 cache: all 100k frames occupy about 1.23 GB uncompressed. Load at most the current clip and 128-image recent window as float tensors; do not load the whole 100k stream as float32. Fetch new videos one at a time, record provenance, decode/check them, and remove only newly created temporary raw downloads after successful cache verification. Preserve existing archives and earlier results.

N166 completed 304,640 arrivals in about 5.1 hours. At that throughput, 720k arrivals alone take about 12 hours. Initial planning range is 12-20 hours after intake, including offline assays, more frequent health measurement and probes; replace it with a measured estimate after a timed pilot. Intake/download time is additional and unmeasured. Keep a 3 GiB disk reserve; current free space is about 17 GiB. No deletion is needed for the design phase.

The eventual notebook refreshes every 15 seconds and includes actual incoming/trained-image loss, actual local/audit RankMe and variance, H1 forecast versus actual, forecast RMSE versus persistence, linear probe, action/fallback counts, measured preview/assay/training times, disk use and ETA. Every 250 arrivals atomically checkpoint learner, optimizer, forecasters, issued pending forecast, ring IDs, quota/RNG state and ledger. Verify interrupted recovery against an uninterrupted pilot. Use a Windows task and watchdog, prevent idle sleep and resume checkpoints after unexpected exits. Forced sleep still suspends computation until wake.

Launch prerequisites are a verified data allocation, a common-warm/holdout exclusion check, H1 no-future-access and panel-identity checks, exact quota checks, predictor predict-before-learn checks, and the timed checkpoint-recovery pilot. This document is the design; it does not start those long jobs.


## Sources

- Local: N97 notebook; N165 predictor screen; N166 comparison.json and notebook.
- BDD100K official download documentation: https://github.com/bdd100k/bdd100k/blob/master/doc/source/download.rst
- Garrido et al., RankMe (ICML 2023): https://arxiv.org/abs/2210.02885 . This motivates measuring effective rank and validating downstream association; it does not establish that maximizing local MAE RankMe improves next-image loss or scene-probe accuracy.
